# GPU Parallelization of Casadi Functions

In [ ]:
# Setup parallelization parameters
BATCH_SIZE = 1024    # Number of function evaluations to parallelize
PRECISION = 'float'  # 'float' or 'double'
FUNCTIONS = 'all'    # List of casadi function to be parallelized
# FUNCTIONS = ['fn_a', 'fn_b' ...]  # Specify function names to parallelize

In [7]:
from cusadi.parallelization import parallelize

# This function generates and compiles JIT CUDA kernels for all specified Casadi functions.
# Regenerates and recompiles the functions even if no changes were made.
# Only needs to be called for modified or new functions (avoid needless recompilation).
parallelize(fn_names=FUNCTIONS, batch_size=BATCH_SIZE, precision=PRECISION)

Loaded CasADi function: ADMM_step (24522 instructions)
Loaded CasADi function: tau (200 instructions)
Generating CUDA code for CasADi function:  ADMM_step
     Number of instructions:  24522
     Number of inputs:  6
     Number of outputs:  3
     Number of work variables:  1194
     Batch size:  1024
     Precision:  float
CUDA codegen complete for ADMM_step.
Kernel written to /home/sehwan/Research/cusadi/cusadi/parallelization/codegen/ADMM_step.cu
Pybind complete for ADMM_step
Binding written to /home/sehwan/Research/cusadi/cusadi/parallelization/codegen/bindings.cpp
Generating CUDA code for CasADi function:  tau
     Number of instructions:  200
     Number of inputs:  3
     Number of outputs:  1
     Number of work variables:  9
     Batch size:  1024
     Precision:  float
CUDA codegen complete for tau.
Kernel written to /home/sehwan/Research/cusadi/cusadi/parallelization/codegen/tau.cu
Pybind complete for tau
Binding written to /home/sehwan/Research/cusadi/cusadi/parallelizatio

No modifications detected for re-loaded extension module cusadi_kernels, skipping build step...
Loading extension module cusadi_kernels...


In [ ]:
import os
import casadi as ca
from cusadi import FUNCTION_DIR
from cusadi.parallelization import CusadiFunction

# Builds CUDA kernels with bindings and loads JIT libraries for use with CusadiFunction.
test_casadi_fn = ca.Function.load(os.path.join(FUNCTION_DIR, 'ADMM_step.casadi'))
test_cusadi_fn = CusadiFunction(test_casadi_fn, batch_size=BATCH_SIZE)
test_cusadi_fn.test(BATCH_SIZE)

Loading JIT kernels from:  /home/sehwan/Research/cusadi/cusadi/parallelization/codegen
Build directory:  /home/sehwan/Research/cusadi/build


Detected CUDA files, patching ldflags
Emitting ninja build file /home/sehwan/Research/cusadi/build/build.ninja...
/home/sehwan/miniconda3/envs/env_cusadi/lib/python3.11/site-packages/torch/utils/cpp_extension.py:2356: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module cusadi_kernels...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


ninja: no work to do.
Loaded CasADi function:  ADMM_step:(x_linsys_soln[1908],z_lb_sym[1188],z_ub_sym[1188],x_k[720],y_k[1188],z_k[1188])->(x_next[720],y_next[1188],z_next[1188]) SXFunction
Loaded library:  <built-in method ADMM_step of PyCapsule object at 0x77e09531a2e0>
Checking input dimensions...
    Input tensor sizes:  [torch.Size([1024, 1908]), torch.Size([1024, 1188]), torch.Size([1024, 1188]), torch.Size([1024, 720]), torch.Size([1024, 1188]), torch.Size([1024, 1188])]
    Output tensor sizes:  [torch.Size([1024, 720]), torch.Size([1024, 1188]), torch.Size([1024, 1188])]
    Work tensor size:  torch.Size([1194, 1024])
Time taken for 1024 environments (GPU): 0.000953399 seconds.


Loading extension module cusadi_kernels...


Time taken for 1024 environments (CPU): 1.683118518 seconds.
Average error for each environment:
    Output 0: Average error norm/env for 1024 envs.: 1.2269008986047534e-09
    Output 1: Average error norm/env for 1024 envs.: 6.901369190641527e-09
    Output 2: Average error norm/env for 1024 envs.: 4.785239595569582e-10
